In [10]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [11]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [12]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [13]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [14]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [15]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [16]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [17]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 47,
 'tn': 2635,
 'fp': 2,
 'fn': 316,
 'misclassification_rate': 0.106,
 'false_positive_rate': 0.0007584376185058779,
 'false_negative_rate': 0.8705234159779615}

### Check results on the test set (new data not yet seen by the model)

In [18]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 9,
 'tn': 873,
 'fp': 1,
 'fn': 117,
 'misclassification_rate': 0.118,
 'false_positive_rate': 0.0011441647597254005,
 'false_negative_rate': 0.9285714285714286}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

I didnt know i had to click restart after a change so by the time i actually tested my changes i had all the parameters set pretty heavy. It took 5m 52.7s to train the model. I was able to get the miscalculation rate pretty low. Lets just say Rick's job would be a lot easier if he had this.

### What are potential ramifications of false positives from the model?

In this usecase, we would unknowingly ban alot of innocent users, which would harm us directly by reducing the userbase while also reducing trust in the company and platform.

### What are potential ramifications of false negatives from the model?

The whole purpose of the model is to detect bots. Not banning bots would waste the same amount of resources without solving any of the problems. An influx of bots would cause the platform to be bloated with fake content that would slowly drive out the human userbase